# 02b: Multi-Seed + Parameter Sweep (Multi-Seed & Parameter Sweep)

**Cold-Start Social-Graph Leakage: Robustness Appendix**

This notebook converts the *single-seed* result in the paper into the **distribution** that the claim ("emergent property") requires:

1. **Multi-seed band**: mean ± confidence interval of cold/warm precision, recall, and attacker advantage across a large number of seeds at the baseline configuration (private 0.30, homophily 0.70, reciprocity 0.49).
2. **OFAT parameter sweep**: tests the independence of the effect from graph parameters by sweeping `private_ratio`, `homophily`, `reciprocity` one at a time (one-factor-at-a-time).

**Design fidelity:** The model code (`LightGCN`, `FeatureGNN`, `train_bpr`, `link_auc`, gated `Recommender`) is copied **verbatim** from 02. The non-circularity rule is preserved: no "recommend the target's neighbor" rule is placed anywhere; the leakage again emerges spontaneously from the cold-start context-fallback.

**Self-contained:** It does not depend on 01's Drive data; it generates the graphs internally using 01's schema (`SocialGraphDataset`), so we can sweep the parameters freely. *(If you want to use your own 01 generator, replace the `make_sbm` / `make_lfr` calls with it.)*

> **Output:** When the run finishes, the cell pr

## 0. Kurulum

In [ ]:
!pip install -q networkx numpy pandas scipy torch

In [ ]:
# --- repository-root anchor (added when notebooks moved into notebooks/) ---
# Notebooks live in notebooks/ but read and write cold_start_leakage/ at the
# repository root, so resolve that root explicitly instead of relying on cwd.
import os as _os_anchor
_here = _os_anchor.getcwd()
if _os_anchor.path.basename(_here) == 'notebooks':
    _os_anchor.chdir(_os_anchor.path.dirname(_here))
# --- end anchor ---
import os, json, time, math
import numpy as np, pandas as pd, networkx as nx
import torch, torch.nn as nn
from scipy.stats import wilcoxon

def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/cold_start_leakage'
else:
    ROOT = os.path.abspath('./cold_start_leakage')
OUT_DIR = os.path.join(ROOT, 'sweeps')
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # required for large-scale (n~15K)
print("Output directory:", OUT_DIR)
print("torch", torch.__version__, "| device:", DEVICE)

## 1. Scan settings

`MODE = "quick"` is for a quick smoke test (a few minutes) first; verify that everything works, then set `MODE = "full"`. You can freely modify the grids below.

In [ ]:
MODE = "quick"          # "quick" (fast test) | "full" (the run reported in the paper)

BASELINE = dict(private_ratio=0.30, homophily=0.70, reciprocity=0.49)

# (B) NEGATIVE CONTROL axes - FLAT output is EXPECTED for LightGCN:
#   reciprocity: norm_adj erases it via G.to_undirected() -> same matrix
#   homophily  : LightGCN does not use node features -> no effect
#   private_ratio: does not change the structure, only selects the target subset
# Flat curves are the sanity check that the harness does not manufacture signal from a nuisance parameter.
CONTROL_GRID = {
    "private_ratio": [0.20, 0.40],
    "homophily":     [0.50, 0.90],
    "reciprocity":   [0.30, 0.70],
}

# (C) INFORMATIVE: the density axis that actually drives the undirected structure (specific to the graph)
DENSITY_GRID = {
    "sbm": ("p_in", [0.04, 0.06, 0.08]),   # intra-community connection probability
    "lfr": ("mu",   [0.05, 0.10, 0.20]),   # community mixing parameter
}

# (D) FEATURE SENSITIVITY: sweep homophily over FeatureGNN (feature channel is active here)
FEATURE_HOMOPHILY = [0.30, 0.50, 0.70, 0.90]

# (E) FORMULA SENSITIVITY (Reviewer 3, item 2): changing the cold-to-warm gate/fusion formula and
# tests whether the leakage depends on this exact formula or on the mechanism (cold = pure context fallback).
# Pretrained LightGCN embeddings are REUSED (NO retraining) -> practically free.
# Baseline values (the no-change row already comes from the multi-seed band): gate=rational, k_gate=K_GATE, fusion=convex
FORMULA_GATES  = ["rational", "exponential", "step"]      # changes the shape of the alpha(h) saturation
FORMULA_KGATES = [2, 3, 5, 10, 20]                          # changes the scale of the rational gate
FORMULA_FUSION = ["convex", "embed_blend", "max"]           # how observer and target scores are combined

if MODE == "quick":
    MS_SEEDS    = [0, 1, 2, 3]
    OFAT_SEEDS  = [0, 1, 2]
    GRAPHS      = ["sbm"]
    MS_MODELS   = ["lightgcn", "mf_bpr"]
    EPOCHS      = 30
else:  # full
    MS_SEEDS    = list(range(10))
    OFAT_SEEDS  = [0, 1, 2, 3, 4]
    GRAPHS      = ["sbm", "lfr"]
    MS_MODELS   = ["lightgcn", "featuregnn", "mf_bpr"]
    EPOCHS      = 45

K, K_GATE, H_WARM, FEAT_DIM, N_NODES = 20, 5, 15, 16, 700
print(f"MODE={MODE} | multiseed={len(MS_SEEDS)} seed x {len(GRAPHS)} graph x {len(MS_MODELS)} model")

## 2. Graph generation (01 schema; parametric)

Directed follow graph: undirected base (SBM/LFR) → orientation via `reciprocity` (mutual/one-way). Feature vector is linked to the community via `homophily`; nodes are labeled private in proportion `private_ratio`. `N(t)=followers ∪ followed` is known exactly (ground truth).

In [ ]:
class SocialGraphDataset:
    def __init__(s, name, G, private, community, node_list, features, meta=None):
        s.name, s.G, s.private, s.community = name, G, private, community
        s.node_list, s.features, s.meta = node_list, features, (meta or {})
    def followers(s, t):  return set(s.G.predecessors(t))
    def following(s, t):  return set(s.G.successors(t))
    def neighborhood(s, t): return s.followers(t) | s.following(t)   # GROUND TRUTH

def _orient(Gund, reciprocity, rng):
    D = nx.DiGraph(); D.add_nodes_from(Gund.nodes())
    for u, v in Gund.edges():
        if u == v: continue
        if rng.random() < reciprocity:
            D.add_edge(u, v); D.add_edge(v, u)
        else:
            (D.add_edge(u, v) if rng.random() < 0.5 else D.add_edge(v, u))
    return D

def _features(n, n_comm, comm, homophily, feat_dim, rng):
    mu = rng.normal(size=(n_comm, feat_dim))
    X = np.zeros((n, feat_dim))
    for i in range(n):
        X[i] = homophily * mu[comm[i]] + (1 - homophily) * rng.normal(size=feat_dim)
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    return X.astype(np.float32)

def _labels(n, private_ratio, rng):
    k = int(round(private_ratio * n))
    priv = set(rng.choice(n, size=k, replace=False).tolist())
    return {i: (i in priv) for i in range(n)}

def make_sbm(n=700, n_comm=None, p_in=0.06, p_out=0.006,
             reciprocity=0.49, homophily=0.70, private_ratio=0.30,
             feat_dim=16, seed=0):
    # n_comm=None -> ~100 nodes per community kept constant (with old constant value n_comm=7 at n=700
    # gives the exact same result); in large-scale (n>>700) runs, the intra-community density
    # necessary to keep independent of n (see the p_out comment in the large-scale cell below).
    if n_comm is None: n_comm = max(2, round(n / 100))
    rng = np.random.default_rng(seed)
    sizes = [n // n_comm] * n_comm
    for i in range(n - sum(sizes)): sizes[i] += 1
    probs = np.full((n_comm, n_comm), p_out); np.fill_diagonal(probs, p_in)
    Gund = nx.stochastic_block_model(sizes, probs, seed=int(seed))
    comm = {i: Gund.nodes[i]['block'] for i in Gund.nodes()}
    D = _orient(Gund, reciprocity, rng)
    X = _features(n, n_comm, comm, homophily, feat_dim, rng)
    return SocialGraphDataset("sbm", D, _labels(n, private_ratio, rng), comm,
                              list(range(n)), X, {"directed_edges": D.number_of_edges()})

def make_lfr(n=700, reciprocity=0.49, homophily=0.70, private_ratio=0.30,
             feat_dim=16, seed=0, mu=0.1):
    rng = np.random.default_rng(seed); Gund = None
    for off in range(8):
        try:
            Gund = nx.LFR_benchmark_graph(n=n, tau1=3, tau2=1.5, mu=mu,
                    average_degree=10, min_community=30, seed=int(seed)+off, max_iters=500)
            break
        except Exception:
            continue
    if Gund is None:
        return None
    Gund = nx.Graph(Gund); Gund.remove_edges_from(nx.selfloop_edges(Gund))
    Gund = nx.convert_node_labels_to_integers(Gund)
    raw = {i: frozenset(Gund.nodes[i]["community"]) for i in Gund.nodes()}
    uniq = {c: j for j, c in enumerate(set(raw.values()))}
    comm = {i: uniq[raw[i]] for i in Gund.nodes()}; n_comm = len(uniq)
    D = _orient(Gund, reciprocity, rng)
    X = _features(n, n_comm, comm, homophily, feat_dim, rng)
    return SocialGraphDataset("lfr", D, _labels(n, private_ratio, rng), comm,
                              list(range(n)), X, {"directed_edges": D.number_of_edges()})

## 3. Model code, copied verbatim from notebook 02 (LightGCN / FeatureGNN / BPR / link-AUC / Recommender)

In [ ]:
def norm_adj(G, n):
    und = G.to_undirected()
    edges = list(und.edges())
    A = torch.zeros((n, n))
    if edges:
        eu, ev = zip(*edges)
        eu = torch.tensor(eu, dtype=torch.long); ev = torch.tensor(ev, dtype=torch.long)
        A[eu, ev] = 1.0; A[ev, eu] = 1.0            # vektorize (eski: Python donguisiyle tek tek)
    A += torch.eye(n)
    deg = A.sum(1); dinv = torch.pow(deg, -0.5); dinv[torch.isinf(dinv)] = 0
    Ahat = A * dinv.unsqueeze(1) * dinv.unsqueeze(0)   # D^-1/2 (A+I) D^-1/2 by broadcasting (not a dense diagonal matmul)
    return Ahat.to(DEVICE)

class LightGCN(nn.Module):
    def __init__(self, n, dim=64, layers=3):
        super().__init__()
        self.emb = nn.Embedding(n, dim); nn.init.normal_(self.emb.weight, std=0.1)
        self.layers = layers
    def forward(self, Ahat):
        e = self.emb.weight; outs = [e]
        for _ in range(self.layers):
            e = Ahat @ e; outs.append(e)
        return torch.stack(outs, 0).mean(0)

class FeatureGNN(nn.Module):
    def __init__(self, n, feat_dim, dim=64, layers=2):
        super().__init__()
        self.id_emb = nn.Embedding(n, dim); nn.init.normal_(self.id_emb.weight, std=0.1)
        self.feat_proj = nn.Linear(feat_dim, dim)
        self.W = nn.ModuleList([nn.Linear(2*dim, dim) for _ in range(layers)])
        self.act = nn.ReLU(); self.layers = layers
    def forward(self, Ahat, Xfeat):
        h = self.id_emb.weight + self.feat_proj(Xfeat)
        for l in range(self.layers):
            agg = Ahat @ h
            h = self.act(self.W[l](torch.cat([h, agg], 1)))
        return h

class MFBaseline(nn.Module):
    # Classic BPR matrix factorization: plain learnable node embeddings, NO graph propagation
    # (Error is accepted but ignored - to comply with train_bpr's call signature).
    # Purpose (Reviewer 3, item 4): to show that the leakage is not specific to GNN message-passing, but that the plain BPR objective's
    # to show it can also arise from placing connected nodes close together in embedding space.
    def __init__(self, n, dim=64):
        super().__init__()
        self.emb = nn.Embedding(n, dim); nn.init.normal_(self.emb.weight, std=0.1)
    def forward(self, Ahat):
        return self.emb.weight

def train_bpr(model, Ahat, pos_pairs, n_nodes, Xfeat=None, epochs=60, bs=1024, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    pos = torch.tensor(pos_pairs, dtype=torch.long, device=DEVICE)
    for ep in range(epochs):
        perm = torch.randperm(pos.size(0), device=DEVICE)
        for i in range(0, pos.size(0), bs):
            idx = perm[i:i+bs]; u, p = pos[idx, 0], pos[idx, 1]
            neg = torch.randint(0, n_nodes, (u.size(0),), device=DEVICE)
            E = model(Ahat) if Xfeat is None else model(Ahat, Xfeat)
            su = (E[u]*E[p]).sum(1); sn = (E[u]*E[neg]).sum(1)
            loss = -torch.log(torch.sigmoid(su - sn) + 1e-9).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (model(Ahat) if Xfeat is None else model(Ahat, Xfeat)).detach()

def link_auc(E, G, n_samples=2000, rng=None):
    rng = rng or np.random.default_rng(0)
    und = G.to_undirected(); edges = list(und.edges()); nodes = list(G.nodes())
    pos = [edges[i] for i in rng.choice(len(edges), size=min(n_samples, len(edges)), replace=False)]
    En = torch.nn.functional.normalize(E, dim=1)
    sc = lambda a, b: float((En[a]*En[b]).sum())
    ps = np.array([sc(u, v) for u, v in pos]); ns = []
    while len(ns) < len(ps):
        a, b = rng.choice(nodes, 2, replace=False)
        if not und.has_edge(a, b): ns.append(sc(a, b))
    ns = np.array(ns)
    return float((ps[:, None] > ns[None, :]).mean())

class Recommender:
    # gate: shape of the alpha(h) saturation | fusion: how observer and target scores combine
    # The defaults (gate="rational", fusion="convex") reproduce the ORIGINAL formula exactly.
    def __init__(self, E, node_list, k_gate=5, gate="rational", fusion="convex"):
        self.E = torch.nn.functional.normalize(torch.as_tensor(E, dtype=torch.float32), dim=1)
        self.node_list = node_list; self.k_gate = k_gate
        self.gate, self.fusion = gate, fusion
    def alpha(self, h):
        if self.gate == "rational":    return h / (h + self.k_gate)              # orijinal formul
        if self.gate == "exponential": return 1 - math.exp(-h / self.k_gate)      # different saturation shape egrisi
        if self.gate == "step":        return 0.0 if h < self.k_gate else 1.0     # sert esik
        raise ValueError(f"bilinmeyen gate: {self.gate}")
    def observer_embedding(self, contacts):
        if len(contacts) == 0: return None
        return torch.nn.functional.normalize(self.E[list(contacts)].mean(0, keepdim=True), dim=1)[0]
    def suggest(self, target, observer_contacts, k=20, exclude=None):
        h = len(observer_contacts); a = self.alpha(h)
        ctx = self.E @ self.E[target]
        if h == 0:
            score = ctx                                             # cold: pure fallback, identical in ALL variants
        elif self.fusion == "convex":
            obs = self.E @ self.observer_embedding(observer_contacts)
            score = a*obs + (1-a)*ctx                                # orijinal formul
        elif self.fusion == "embed_blend":
            q = torch.nn.functional.normalize(
                a*self.observer_embedding(observer_contacts) + (1-a)*self.E[target], dim=0)
            score = self.E @ q                                       # blend the embedding first, then score once
        elif self.fusion == "max":
            obs = self.E @ self.observer_embedding(observer_contacts)
            score = torch.maximum(a*obs, (1-a)*ctx)                  # non-convex birlestirici
        else:
            raise ValueError(f"bilinmeyen fusion: {self.fusion}")
        score = score.clone(); score[target] = -1e9
        for x in (exclude or []): score[x] = -1e9
        return torch.topk(score, k).indices.tolist()

## 4. Measurement: cold vs warm + 3 baselines (mirrors Table II)

For each hidden target, cold/warm precision@K, cold recall@K; advantage over random/popularity/same-community baselines; and paired Wilcoxon p-value for cold-vs-warm.

In [ ]:
def measure(ds, E, k=K, k_gate=K_GATE, h_warm=H_WARM, seed=0, gate="rational", fusion="convex"):
    rec = Recommender(E, ds.node_list, k_gate=k_gate, gate=gate, fusion=fusion)
    rng = np.random.default_rng(seed + 9991)
    nodes = ds.node_list
    targets = [t for t in nodes if ds.private[t] and ds.G.degree(t) >= 4]
    deg = dict(ds.G.degree()); deg_sorted = sorted(nodes, key=lambda x: deg[x], reverse=True)
    by_comm = {}
    for x in nodes: by_comm.setdefault(ds.community[x], []).append(x)
    cold_p, cold_r, warm_p, b_rand, b_pop, b_comm = [], [], [], [], [], []
    for t in targets:
        Nt = ds.neighborhood(t)
        if not Nt: continue
        s_cold = set(rec.suggest(t, set(), k=k))
        cold_p.append(len(s_cold & Nt)/k); cold_r.append(len(s_cold & Nt)/len(Nt))
        wc = set(rng.choice(nodes, h_warm, replace=False).tolist()) - Nt - {t}
        warm_p.append(len(set(rec.suggest(t, wc, k=k)) & Nt)/k)
        rr = set(rng.choice(nodes, k, replace=False).tolist()) - {t}
        b_rand.append(len(rr & Nt)/k)
        b_pop.append(len(set([x for x in deg_sorted if x != t][:k]) & Nt)/k)
        same = [x for x in by_comm[ds.community[t]] if x != t]
        ss = set(rng.choice(same, min(k, len(same)), replace=False).tolist()) if same else set()
        b_comm.append(len(ss & Nt)/k)
    cp, wp = np.array(cold_p), np.array(warm_p)
    try:
        p = wilcoxon(cp, wp).pvalue if len(cp) and np.any(cp != wp) else float('nan')
    except Exception:
        p = float('nan')
    best_base = max(np.mean(b_rand), np.mean(b_pop), np.mean(b_comm))
    return dict(n_targets=len(cold_p), cold_prec=float(np.mean(cold_p)),
                cold_recall=float(np.mean(cold_r)), warm_prec=float(np.mean(wp)),
                base_random=float(np.mean(b_rand)), base_pop=float(np.mean(b_pop)),
                base_comm=float(np.mean(b_comm)),
                advantage=float(np.mean(cp) - best_base), wilcoxon_p=float(p))

def _train(graph_type, params, model_name, seed, epochs=EPOCHS, feat_dim=FEAT_DIM, n_nodes=None):
    # Graph generation and model training. run_one() and the formula-sensitivity block (E) share this,
    # so the formula variants (on the same E) run WITHOUT RETRAINING.
    # n_nodes=None -> global N_NODES (unchanged behavior at all current call sites); large-scale
    # cell overrides it explicitly (for example n_nodes=15000).
    torch.manual_seed(seed); np.random.seed(seed)
    n_use = N_NODES if n_nodes is None else n_nodes
    ds = (make_sbm(n=n_use, seed=seed, feat_dim=feat_dim, **params) if graph_type == "sbm"
          else make_lfr(n=n_use, seed=seed, feat_dim=feat_dim, **params))
    if ds is None: return None
    n = len(ds.node_list)
    Ahat = norm_adj(ds.G, n)                         # already returns on .to(DEVICE)
    pos_pairs = [(u, v) for u, v in ds.G.to_undirected().edges()]
    if model_name == "lightgcn":
        mdl = LightGCN(n, dim=64, layers=3).to(DEVICE); E = train_bpr(mdl, Ahat, pos_pairs, n, epochs=epochs)
    elif model_name == "mf_bpr":
        mdl = MFBaseline(n, dim=64).to(DEVICE); E = train_bpr(mdl, Ahat, pos_pairs, n, epochs=epochs)
    elif model_name == "featuregnn":
        Xf = torch.tensor(ds.features, dtype=torch.float32, device=DEVICE)
        mdl = FeatureGNN(n, ds.features.shape[1], dim=64, layers=2).to(DEVICE)
        E = train_bpr(mdl, Ahat, pos_pairs, n, Xfeat=Xf, epochs=epochs)
    else:
        raise ValueError(f"bilinmeyen model_name: {model_name}")
    E = E.cpu()                                       # so the downstream path (measure/Recommender/CSV) stays entirely device-independent
    auc = link_auc(E, ds.G, rng=np.random.default_rng(seed))
    return ds, E, auc

def run_one(graph_type, params, model_name, seed, epochs=EPOCHS, feat_dim=FEAT_DIM, n_nodes=None, return_artifacts=False):
    out = _train(graph_type, params, model_name, seed, epochs, feat_dim, n_nodes=n_nodes)
    if out is None: return (None, None, None) if return_artifacts else None
    ds, E, auc = out
    m = measure(ds, E.numpy(), seed=seed)
    row = dict(graph_type=graph_type, model=model_name, seed=seed, link_auc=round(auc, 4),
               directed_edges=ds.meta.get("directed_edges"), **params)
    row.update(m)
    return (row, ds, E) if return_artifacts else row

## 5. Duration estimation (ETA) and run the scan

First it times a single run, estimates the total duration, and then executes the entire scan. Progress is printed line by line; if the run crashes, `results_long.csv` preserves the rows accumulated up to that point.

In [ ]:
# single-run calibration -> ETA
_t = time.time(); _ = run_one(GRAPHS[0], BASELINE, MS_MODELS[0], MS_SEEDS[0]); per_run = time.time() - _t

ms_runs      = len(MS_SEEDS) * len(GRAPHS) * len(MS_MODELS)
control_runs = sum(len(v) for v in CONTROL_GRID.values()) * len(OFAT_SEEDS) * len(GRAPHS)
density_runs = sum(len(vals) for (_p, vals) in DENSITY_GRID.values() if _p) * len(OFAT_SEEDS) \
               if False else sum(len(DENSITY_GRID[gt][1]) for gt in GRAPHS) * len(OFAT_SEEDS)
feature_runs = len(FEATURE_HOMOPHILY) * len(OFAT_SEEDS) * len(GRAPHS)
total = ms_runs + control_runs + density_runs + feature_runs
print(f"~{per_run:.1f}s/run | multiseed={ms_runs} + control={control_runs} + "
      f"density={density_runs} + feature={feature_runs} = {total} runs")
print(f"Estimated total time: ~{per_run*total/60:.1f} minutes")

In [ ]:
rows, done, t0 = [], 0, time.time()

def _log(r, total):
    global done; done += 1
    tag = r.get("swept_param", "none")
    print(f"[{done:3d}/{total}] {r['block']:9s} {r['graph_type']:4s} {r['model']:10s} "
          f"seed={r['seed']} {tag} auc={r['link_auc']:.3f} cold={r['cold_prec']:.3f} "
          f"warm={r['warm_prec']:.3f} recall={r['cold_recall']:.3f} adv={r['advantage']:+.3f}")

def _save(): pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "results_long.csv"), index=False)

# (A) multi-seed band - base config (cached for LightGCN embeddings (E) block)
_EMB_CACHE = {}   # (graph_type, seed) -> (ds, E, auc); lightgcn only, for the formula-sensitivity block
for seed in MS_SEEDS:
    for gt in GRAPHS:
        for mdl in MS_MODELS:
            r, ds, E = run_one(gt, dict(BASELINE), mdl, seed, return_artifacts=True)
            if r:
                r.update(block="multiseed", swept_param="none"); rows.append(r); _log(r, total); _save()
                if mdl == "lightgcn": _EMB_CACHE[(gt, seed)] = (ds, E, r["link_auc"])

# (B) NEGATIVE CONTROL - LightGCN, flat expected
for param, values in CONTROL_GRID.items():
    for v in values:
        cfg = dict(BASELINE); cfg[param] = v
        for seed in OFAT_SEEDS:
            for gt in GRAPHS:
                r = run_one(gt, cfg, "lightgcn", seed)
                if r: r.update(block="control", swept_param=param); rows.append(r); _log(r, total); _save()

# (C) INFORMATIVE density, plays the undirected structure (LightGCN)
for gt in GRAPHS:
    pname, vals = DENSITY_GRID[gt]
    for v in vals:
        cfg = dict(BASELINE); cfg[pname] = v       # p_in (sbm) / mu (lfr)
        for seed in OFAT_SEEDS:
            r = run_one(gt, cfg, "lightgcn", seed)
            if r: r.update(block="density", swept_param=pname, density_val=v); rows.append(r); _log(r, total); _save()

# (D) FEATURE sensitivity, homophily x FeatureGNN
for v in FEATURE_HOMOPHILY:
    cfg = dict(BASELINE); cfg["homophily"] = v
    for seed in OFAT_SEEDS:
        for gt in GRAPHS:
            r = run_one(gt, cfg, "featuregnn", seed)
            if r: r.update(block="feature", swept_param="homophily"); rows.append(r); _log(r, total); _save()

# (E) FORMULA sensitivity, rescore pre-trained LightGCN embeddings WITHOUT RETRAINING
# (Reviewer 3, item 2). Baseline row (gate=rational,k_gate=K_GATE,fusion=convex) already exists in (A),
# here only ONE axis deviates from baseline value (orthogonal OFAT, no mixed effect).
_f_before = len(rows)
for (gt, seed), (ds, E, auc) in _EMB_CACHE.items():
    En = E.numpy()
    for gate in FORMULA_GATES:
        if gate == "rational": continue
        m = measure(ds, En, seed=seed, gate=gate, fusion="convex", k_gate=K_GATE)
        rows.append(dict(graph_type=gt, model="lightgcn", seed=seed, link_auc=auc,
                          block="formula", swept_param="gate", formula_value=gate, **m))
    for kg in FORMULA_KGATES:
        if kg == K_GATE: continue
        m = measure(ds, En, seed=seed, gate="rational", fusion="convex", k_gate=kg)
        rows.append(dict(graph_type=gt, model="lightgcn", seed=seed, link_auc=auc,
                          block="formula", swept_param="k_gate", formula_value=kg, **m))
    for fus in FORMULA_FUSION:
        if fus == "convex": continue
        m = measure(ds, En, seed=seed, gate="rational", fusion=fus, k_gate=K_GATE)
        rows.append(dict(graph_type=gt, model="lightgcn", seed=seed, link_auc=auc,
                          block="formula", swept_param="fusion", formula_value=fus, **m))
_save()
print(f"[formula] {len(rows)-_f_before} satir eklendi (yeniden egitim yok, {len(_EMB_CACHE)} onbellek gommeden)")

df = pd.DataFrame(rows); _save()
print(f"\nBitti: {len(df)} satir, {time.time()-t0:.1f}s -> {OUT_DIR}/results_long.csv")
df.head()

## 6. Aggregate and write the run summary

The cell below produces two tables:
- **Multi-seed band:** cold/warm/recall/advantage/AUC average per graph×model ± 95% CI (across seeds).
- **OFAT:** cold precision average ± std per parameter×value×graph.

Everything is written to `RUN_SUMMARY.md` **and** printed below. You can copy the text between the two `=====` lines and send it to me.

In [ ]:
def ci95(x):
    x = np.asarray(x, float); n = len(x)
    return 1.96 * x.std(ddof=1) / math.sqrt(n) if n > 1 else float('nan')

df = pd.read_csv(os.path.join(OUT_DIR, "results_long.csv"))
ms  = df[df.block == "multiseed"]
ctl = df[df.block == "control"]
den = df[df.block == "density"]
fea = df[df.block == "feature"]
frm = df[df.block == "formula"] if "formula" in set(df.block.unique()) else df.iloc[0:0]

L = ["### 1) Multi-seed band (baseline: private=0.30, homophily=0.70, reciprocity=0.49)", "",
     "| graph | model | n_seed | cold_prec (mu+-CI) | warm_prec (mu+-CI) | cold_recall (mu+-CI) | advantage (mu+-CI) | link_auc | medyan p |",
     "|---|---|---|---|---|---|---|---|---|"]
for (gt, mdl), g in ms.groupby(["graph_type", "model"]):
    mc = lambda c: f"{g[c].mean():.3f} +- {ci95(g[c]):.3f}"
    L.append(f"| {gt} | {mdl} | {len(g)} | {mc('cold_prec')} | {mc('warm_prec')} | "
             f"{mc('cold_recall')} | {mc('advantage')} | {g['link_auc'].mean():.3f} | {g['wilcoxon_p'].median():.1e} |")

L += ["", "### 2) NEGATIVE CONTROL (LightGCN; cold_prec mu+-std) -- FLAT expected = harness healthy", "",
      "| parameter | value | graph | n | cold_prec | warm_prec |", "|---|---|---|---|---|---|"]
base_lg = {gt: ms[(ms.graph_type==gt)&(ms.model=='lightgcn')] for gt in df.graph_type.unique()}
for param in CONTROL_GRID:
    for gt in sorted(ctl.graph_type.unique()):
        b = base_lg.get(gt)
        if b is not None and len(b):
            L.append(f"| {param} | {BASELINE[param]} (taban) | {gt} | {len(b)} | "
                     f"{b['cold_prec'].mean():.3f} +- {b['cold_prec'].std(ddof=1):.3f} | {b['warm_prec'].mean():.3f} |")
        for v, g in ctl[(ctl.swept_param==param)&(ctl.graph_type==gt)].groupby(param):
            L.append(f"| {param} | {v} | {gt} | {len(g)} | "
                     f"{g['cold_prec'].mean():.3f} +- {g['cold_prec'].std(ddof=1):.3f} | {g['warm_prec'].mean():.3f} |")

L += ["", "### 3) INFORMATIONAL: structural density (LightGCN; cold_prec mu+-std) -- the slope here is the real mechanism", "",
      "| graph | density param | value | n | cold_prec | cold_recall | warm_prec | link_auc |",
      "|---|---|---|---|---|---|---|---|"]
if len(den):
    for (gt, pname), g0 in den.groupby(["graph_type", "swept_param"]):
        for v, g in g0.groupby("density_val"):
            L.append(f"| {gt} | {pname} | {v} | {len(g)} | "
                     f"{g['cold_prec'].mean():.3f} +- {g['cold_prec'].std(ddof=1):.3f} | "
                     f"{g['cold_recall'].mean():.3f} | {g['warm_prec'].mean():.3f} | {g['link_auc'].mean():.3f} |")

L += ["", "### 4) FEATURE sensitivity: homophily x FeatureGNN (cold_prec mu+-std)", "",
      "| graph | homophily | n | cold_prec | warm_prec | link_auc |", "|---|---|---|---|---|---|"]
if len(fea):
    for (gt, v), g in fea.groupby(["graph_type", "homophily"]):
        L.append(f"| {gt} | {v} | {len(g)} | {g['cold_prec'].mean():.3f} +- {g['cold_prec'].std(ddof=1):.3f} | "
                 f"{g['warm_prec'].mean():.3f} | {g['link_auc'].mean():.3f} |")

L += ["", "### 5) FORMULA sensitivity (Referee 3.2): base=rational/k_gate=5/convex, NO RETRAINING", "",
      "| axis | value | graph | n | cold_prec | cold_recall | advantage |", "|---|---|---|---|---|---|---|"]
if len(frm):
    for (axis, val, gt), g in frm.groupby(["swept_param", "formula_value", "graph_type"]):
        L.append(f"| {axis} | {val} | {gt} | {len(g)} | "
                 f"{g['cold_prec'].mean():.3f} +- {g['cold_prec'].std(ddof=1):.3f} | "
                 f"{g['cold_recall'].mean():.3f} | {g['advantage'].mean():+.3f} |")
    L.append("| *(taban: rational / 5 / convex)* | - | - | - | see Table 1 (multiseed, lightgcn) | - | - |")

meta = [f"# Cold-Start: Multi-Seed and Parameter Sweep summary",
        f"- MODE={MODE} | K={K} | k_gate={K_GATE} | h_warm={H_WARM} | epochs={EPOCHS} | n_nodes={N_NODES}",
        f"- total runs: {len(df)} | graphs: {sorted(df.graph_type.unique())}", ""]
share = "\n".join(meta + L) + "\n"
with open(os.path.join(OUT_DIR, "RUN_SUMMARY.md"), "w") as f: f.write(share)
print("=" * 70); print(share); print("=" * 70)
print(f"\n[saved] {OUT_DIR}/RUN_SUMMARY.md  and  results_long.csv")
print("The block between the two ===== lines is the run summary.")

## 7. (Optional) Visuals: multi-seed violin plots + OFAT curves

In [ ]:
import matplotlib.pyplot as plt
df = pd.read_csv(os.path.join(OUT_DIR, "results_long.csv"))
ms = df[df.block == "multiseed"]; den = df[df.block == "density"]; fea = df[df.block == "feature"]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
lg = ms[ms.model == "lightgcn"]
ax[0].violinplot([lg["cold_prec"].values, lg["warm_prec"].values], showmeans=True)
ax[0].set_xticks([1, 2]); ax[0].set_xticklabels(["cold", "warm"]); ax[0].set_ylabel("precision@20")
ax[0].set_title(f"1) Multi-seed distribution (LightGCN, n={len(lg)})")

for gt, g in den.groupby("graph_type"):
    gg = g.groupby("density_val")["cold_prec"].agg(["mean", "std"]).reset_index()
    ax[1].errorbar(gg["density_val"], gg["mean"], yerr=gg["std"], marker="o", capsize=4, label=gt)
ax[1].set_xlabel("density (p_in / mu)"); ax[1].set_ylabel("cold precision@20")
ax[1].set_title("3) Informative: structural density"); ax[1].legend()

for gt, g in fea.groupby("graph_type"):
    gg = g.groupby("homophily")["cold_prec"].agg(["mean", "std"]).reset_index()
    ax[2].errorbar(gg["homophily"], gg["mean"], yerr=gg["std"], marker="s", capsize=4, label=gt)
ax[2].set_xlabel("homophily"); ax[2].set_ylabel("cold precision@20")
ax[2].set_title("4) FeatureGNN: homophily"); ax[2].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "sweep_figs.png"), dpi=130); plt.show()
print("[saved]", os.path.join(OUT_DIR, "sweep_figs.png"))

## 8. (NEW) Large-Scale Stress Test: n~15,000 (Reviewer 3, item 3)

We are testing whether the current leakage effect is also observed at a scale much larger than the
n=700-4,000 range. Taking advantage of `make_sbm`'s `n_comm=None` default updated above, we keep the
community size (~100 nodes) fixed as `n` grows: otherwise, even if intra-community density (`p_in`) stayed
the same, community size would grow and degree would naturally increase.

**A second, more subtle point:** if `p_out` (inter-community connection probability) is left FIXED, a
node's expected inter-community degree becomes `p_out * (n - community_size)` and this grows LINEARLY with
`n` even if community_size is kept fixed (because the number of "other" nodes grows with n). At the n=700
baseline this term is `0.006 * 600 = 3.6`; to preserve the same value at n=15,000 as well, we scale `p_out`
downward. If this were not done, the n=15,000 graph would come out meaninglessly dense (average degree
~90+) and the community structure would be crushed.

Even at this scale, the dense `Ahat` (~n^2*4 byte ~= 900MB, on DEVICE) fits comfortably on a single GPU;
no sparse rewrite is needed (this would only be necessary for n>30-50K, see plan).

In [ ]:
# at n=700 base: n_comm=7, community size=100, p_out=0.006 -> expected inter-community degree = 0.006*(700-100)=3.6
_BASE_N, _BASE_POUT = 700, 0.006
_base_ncomm = max(2, round(_BASE_N/100)); _base_size = _BASE_N/_base_ncomm
_base_inter_deg = _BASE_POUT * (_BASE_N - _base_size)

LARGE_N = 15000
LARGE_SEEDS = [0, 1, 2, 3, 4]
_large_ncomm = max(2, round(LARGE_N/100)); _large_size = LARGE_N/_large_ncomm
_large_p_out = _base_inter_deg / (LARGE_N - _large_size)   # keeps the expected inter-community degree equal to the n=700 baseline

print(f"[large-scale] device={DEVICE} | target n={LARGE_N} | n_comm={_large_ncomm} | "
      f"p_out(olcekli)={_large_p_out:.6f} (taban 0.006 yerine)")

_t = time.time()
_r0, _ds0, _E0 = run_one("sbm", dict(BASELINE, p_out=_large_p_out), "lightgcn", 0,
                          n_nodes=LARGE_N, return_artifacts=True)
_dt = time.time() - _t
if torch.cuda.is_available():
    print(f"[large-scale] single run: {_dt:.1f}s | peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
else:
    print(f"[large-scale] single run: {_dt:.1f}s (CPU)")
print(f"[large-scale] n={len(_ds0.node_list)} | edge={_ds0.meta['directed_edges']} | "
      f"link_auc={_r0['link_auc']:.3f} | cold_prec@20={_r0['cold_prec']:.3f} | warm_prec@20={_r0['warm_prec']:.3f}")
print(f"[large-scale] estimated total time ({len(LARGE_SEEDS)} seed): ~{_dt*len(LARGE_SEEDS)/60:.1f} minutes")
print("[large-scale] if the result is bad, pull LARGE_N down (e.g. 8000-10000) and rerun the cell.")

large_rows = [dict(_r0, block="scale", n_nodes=LARGE_N)]
for seed in LARGE_SEEDS[1:]:
    r = run_one("sbm", dict(BASELINE, p_out=_large_p_out), "lightgcn", seed, n_nodes=LARGE_N)
    if r:
        r.update(block="scale", n_nodes=LARGE_N); large_rows.append(r)
        print(f"  [large-scale seed={seed}] cold={r['cold_prec']:.3f} warm={r['warm_prec']:.3f} "
              f"recall={r['cold_recall']:.3f} auc={r['link_auc']:.3f}")

lg_df = pd.DataFrame(large_rows)
lg_df.to_csv(os.path.join(OUT_DIR, "results_large_scale.csv"), index=False)
print(f"\n[large-scale SUMMARY] n={LARGE_N} | {len(lg_df)} seed | "
      f"cold_prec={lg_df.cold_prec.mean():.3f}+-{ci95(lg_df.cold_prec):.3f} | "
      f"warm_prec={lg_df.warm_prec.mean():.3f}+-{ci95(lg_df.warm_prec):.3f} | "
      f"cold_recall={lg_df.cold_recall.mean():.3f}+-{ci95(lg_df.cold_recall):.3f} | "
      f"link_auc={lg_df.link_auc.mean():.3f}")
print("[saved]", os.path.join(OUT_DIR, "results_large_scale.csv"))